In [ ]:
# Evaluation, Benchmarking, And Integration
# Toutes les 8 parties dans une seule cellule

# PARTIE I : Configuration
import pandas as pd
import torch
import gc
import evaluate
import nltk
from transformers import T5ForConditionalGeneration, AutoTokenizer, GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset

nltk.download('punkt')
nltk.download('punkt_tab')
print("=== PARTIE I : Setup terminé ===")

# PARTIE II : Chargement dataset
print("\n=== PARTIE II : Chargement du dataset ===")
dataset = load_dataset('cnn_dailymail', '3.0.0')
train_sample = dataset['train'].select(range(100))
test_sample = dataset['test'].select(range(50))

train_df = pd.DataFrame({'prompt_text': train_sample['article'], 'prompt_title': train_sample['highlights']})
test_df = pd.DataFrame({'prompt_text': test_sample['article'], 'prompt_title': test_sample['highlights']})

print(f"Train: {len(train_df)}, Test: {len(test_df)}")
print(f"Article: {train_df.iloc[0]['prompt_text'][:150]}...")
print(f"Résumé: {train_df.iloc[0]['prompt_title']}")

# PARTIE III : T5 Summarization
print("\n=== PARTIE III : T5 Summarization ===")

def batch_generator(data, batch_size=8):
    for i in range(0, len(data), batch_size):
        yield data[i:i+batch_size]

def summarize_with_t5(articles, model_name='t5-small', batch_size=8, max_length=150):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    summaries = []
    for batch in batch_generator(articles, batch_size):
        inputs = [f"summarize: {article}" for article in batch]
        encoded = tokenizer(inputs, return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            output_ids = model.generate(encoded['input_ids'], max_length=max_length, num_beams=4, early_stopping=True)
        batch_summaries = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
        summaries.extend(batch_summaries)
        torch.cuda.empty_cache()
        gc.collect()
    del model
    torch.cuda.empty_cache()
    gc.collect()
    return summaries

t5_small_summaries = summarize_with_t5(train_df['prompt_text'].tolist(), 't5-small')
print(f"Generated {len(t5_small_summaries)} summaries")

# PARTIE IV : Accuracy
print("\n=== PARTIE IV : Accuracy ===")
accuracy = sum([1 if pred == ref else 0 for pred, ref in zip(t5_small_summaries, train_df['prompt_title'].tolist())]) / len(t5_small_summaries)
print(f"Accuracy: {accuracy:.4f} (très faible car nécessite correspondance exacte)")

# PARTIE V : ROUGE Implementation
print("\n=== PARTIE V : ROUGE ===")
rouge = evaluate.load('rouge')

def compute_rouge_score(predictions, references):
    def preprocess(text):
        sentences = nltk.sent_tokenize(text)
        return '\n'.join(sentences)
    pred_processed = [preprocess(p) for p in predictions]
    ref_processed = [preprocess(r) for r in references]
    scores = rouge.compute(predictions=pred_processed, references=ref_processed)
    return scores

print("ROUGE metric loaded")

# PARTIE VI : Understanding ROUGE
print("\n=== PARTIE VI : Understanding ROUGE ===")
test1 = compute_rouge_score(["The cat sat on the mat."], ["The cat sat on the mat."])
print(f"Test 1 (exact match): {test1}")
test2 = compute_rouge_score([""], ["The cat sat on the mat."])
print(f"Test 2 (null): {test2}")
test3 = compute_rouge_score(["run quick"], ["running quickly"])
print(f"Test 3 (stemming): {test3}")
test4 = compute_rouge_score(["The quick brown fox."], ["The quick brown fox jumps over the lazy dog."])
print(f"Test 4 (partial): {test4}")
test5a = compute_rouge_score(["A B C"], ["D E F"])
test5b = compute_rouge_score(["D E F"], ["A B C"])
print(f"Test 5 (symmetry): {test5a} vs {test5b}")

# PARTIE VII : Model Comparison
print("\n=== PARTIE VII : Model Comparison ===")

def summarize_with_gpt2(articles, model_name='gpt2', batch_size=8, max_length=100):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    summaries = []
    for batch in batch_generator(articles, batch_size):
        inputs = [f"{article[:500]} TL;DR:" for article in batch]
        encoded = tokenizer(inputs, return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            output_ids = model.generate(encoded['input_ids'], max_length=encoded['input_ids'].shape[1]+max_length, num_beams=4, early_stopping=True, pad_token_id=tokenizer.eos_token_id)
        batch_summaries = []
        for i, ids in enumerate(output_ids):
            full_text = tokenizer.decode(ids, skip_special_tokens=True)
            summary = full_text.split('TL;DR:')[-1].strip()
            batch_summaries.append(summary)
        summaries.extend(batch_summaries)
        torch.cuda.empty_cache()
        gc.collect()
    del model
    torch.cuda.empty_cache()
    gc.collect()
    return summaries

def compute_rouge_per_row(predictions, references):
    rows = []
    for pred, ref in zip(predictions, references):
        scores = compute_rouge_score([pred], [ref])
        rows.append({'rouge1': scores['rouge1'], 'rouge2': scores['rouge2'], 'rougeL': scores['rougeL'], 'prediction': pred, 'reference': ref})
    return pd.DataFrame(rows)

print("Generating t5-base summaries...")
t5_base_summaries = summarize_with_t5(train_df['prompt_text'].tolist(), 't5-base')
print("Generating gpt2 summaries...")
gpt2_summaries = summarize_with_gpt2(train_df['prompt_text'].tolist(), 'gpt2')

t5_small_rouge_df = compute_rouge_per_row(t5_small_summaries, train_df['prompt_title'].tolist())
t5_base_rouge_df = compute_rouge_per_row(t5_base_summaries, train_df['prompt_title'].tolist())
gpt2_rouge_df = compute_rouge_per_row(gpt2_summaries, train_df['prompt_title'].tolist())
print("Per-row ROUGE scores calculated")

# PARTIE VIII : All Models Comparison
print("\n=== PARTIE VIII : All Models Comparison ===")

def compare_models(model_rouge_dfs, model_names):
    comparison_data = []
    for name, df in zip(model_names, model_rouge_dfs):
        avg_scores = {'model': name, 'avg_rouge1': df['rouge1'].mean(), 'avg_rouge2': df['rouge2'].mean(), 'avg_rougeL': df['rougeL'].mean()}
        comparison_data.append(avg_scores)
    return pd.DataFrame(comparison_data)

def compare_models_summaries(summaries_dict, references, num_examples=5):
    comparison_rows = []
    for i in range(min(num_examples, len(references))):
        row = {'reference': references[i]}
        for model_name, summaries in summaries_dict.items():
            row[model_name] = summaries[i]
        comparison_rows.append(row)
    return pd.DataFrame(comparison_rows)

model_comparison = compare_models([t5_small_rouge_df, t5_base_rouge_df, gpt2_rouge_df], ['t5-small', 't5-base', 'gpt2'])
print("\nROUGE Comparison:")
print(model_comparison)

summaries_dict = {'t5-small': t5_small_summaries, 't5-base': t5_base_summaries, 'gpt2': gpt2_summaries}
summary_comparison = compare_models_summaries(summaries_dict, train_df['prompt_title'].tolist(), num_examples=3)
print("\nSummary Examples:")
for idx, row in summary_comparison.iterrows():
    print(f"\n--- Example {idx + 1} ---")
    print(f"Reference: {row['reference']}")
    print(f"T5-small: {row['t5-small']}")
    print(f"T5-base: {row['t5-base']}")
    print(f"GPT2: {row['gpt2']}")

print("\n=== EXERCICE COMPLET - 8 PARTIES TERMINÉES ===")